# RLHF Live Coding Seminar

This notebook is the seminar script: we fill in the TODOs, run the pipeline, and discuss how SFT → Reward Model → PPO with a KL penalty aligns responses.

> Tips: work top to bottom, and don't be afraid to change hyperparameters live during the session. All the tasks are small so we can finish in real time.

## Plan
- Set up the environment and paths
- Generate synthetic data (SFT pairs + preferences)
- Train SFT (reference policy)
- Train a Reward Model on preferences
- Run PPO with a KL penalty and see how the probabilities change
- Play with β and discuss reward hacking

> TODO markers indicate the places to fill in together.

In [ ]:
# (Optional) Install dependencies for a clean environment
# Run this if numpy/matplotlib are missing
# !pip install -q numpy matplotlib

In [ ]:
# Set up paths and imports
from pathlib import Path
import sys

BASE = Path('.').resolve()
if not (BASE / 'simple_text_env.py').exists():
    raise RuntimeError('Run the notebook from the code/15_rlhf_basics directory')

if str(BASE) not in sys.path:
    sys.path.append(str(BASE))

print('BASE:', BASE)

In [ ]:
# TODO-1: Generate synthetic data (SFT pairs + preferences)
from generate_data import main as generate_data

data_dir = BASE / 'data'
# By default this creates 80 preference pairs (num_samples=80).
# You can change this directly in the call:
# generate_data(output_dir=data_dir, num_samples=120)
# or edit the default in generate_data.py
# During the live session, uncomment to generate the data:
# generate_data(output_dir=data_dir, num_samples=80)

data_dir

In [ ]:
# TODO-2: Train SFT (reference policy)
from sft_model import train_sft, TabularPolicy

sft_ckpt = data_dir / 'sft_policy.npy'
# Uncomment to run training (cross-entropy over the tabular policy)
# sft_model = train_sft(sft_path=data_dir / 'sft_data.json', ckpt_path=sft_ckpt)

# Peek at the logits/distributions after training (once a checkpoint exists)
if sft_ckpt.exists():
    sft_model = TabularPolicy.load(sft_ckpt)
    print('SFT probs per prompt:')
    from simple_text_env import PROMPTS, CANDIDATES
    for i, p in enumerate(PROMPTS):
        probs = sft_model.predict_probs(i)
        print(f'Prompt: {p}')
        for c, pr in zip(CANDIDATES, probs):
            print(f'  {pr:0.3f} -> {c}')
        print()
else:
    print('SFT checkpoint not found, train the model above.')

In [ ]:
# TODO-3: Train a Reward Model on preferences
from reward_model import train_reward_model, RewardModel

rm_ckpt = data_dir / 'reward_model.npy'
# Uncomment to train the RM on preferences.json
# rm = train_reward_model(pref_path=data_dir / 'preferences.json', ckpt_path=rm_ckpt)

if rm_ckpt.exists():
    rm = RewardModel.load(rm_ckpt)
    print('Reward weights:', rm.w)
else:
    print('RM checkpoint not found, train the model above.')

In [ ]:
# TODO-4: Run PPO with a KL penalty
from ppo_rlhf import run as run_ppo

beta = 0.01  # Play with this: 0.0, 0.001, 0.01, 0.1
# Uncomment the line below to start RL fine-tuning
# run_ppo(beta=beta, output_dir=data_dir)

print('Ready to run PPO; pick a beta and uncomment the line above.')

In [ ]:
# TODO-5: See how the policy changed after PPO
from simple_text_env import PROMPTS, CANDIDATES
import numpy as np

ppo_ckpt = data_dir / f'ppo_actor_beta_{beta}.npy'
if ppo_ckpt.exists():
    from sft_model import TabularPolicy
    actor = TabularPolicy()
    actor.logits = np.load(ppo_ckpt)
    print(f'Checking ppo_actor_beta_{beta}.npy')
    for i, prompt in enumerate(PROMPTS):
        probs = actor.predict_probs(i)
        print(f'Prompt: {prompt}')
        for resp, p in zip(CANDIDATES, probs):
            print(f'  {p:0.3f} -> {resp}')
        print()
else:
    print('Run PPO above so a checkpoint is created.')

In [ ]:
# Visualization: comparing SFT vs PPO distributions
import numpy as np
import matplotlib.pyplot as plt
from sft_model import TabularPolicy
from simple_text_env import PROMPTS, CANDIDATES

sft_ckpt = data_dir / 'sft_policy.npy'
ppo_ckpt = data_dir / f'ppo_actor_beta_{beta}.npy'

if not sft_ckpt.exists() or not ppo_ckpt.exists():
    print('Need the sft_policy.npy and ppo_actor_beta_{beta}.npy checkpoints. Run TODO-2 and TODO-4.')
else:
    sft = TabularPolicy.load(sft_ckpt)
    ppo = TabularPolicy()
    ppo.logits = np.load(ppo_ckpt)

    fig, axes = plt.subplots(len(PROMPTS), 1, figsize=(8, 10), sharex=True)
    axes = np.atleast_1d(axes)
    x = np.arange(len(CANDIDATES))

    for i, ax in enumerate(axes):
        sft_probs = sft.predict_probs(i)
        ppo_probs = ppo.predict_probs(i)
        ax.bar(x - 0.15, sft_probs, width=0.3, label='SFT')
        ax.bar(x + 0.15, ppo_probs, width=0.3, label='PPO')
        ax.set_title(PROMPTS[i])
        ax.set_ylim(0, 1)
        ax.set_xticks(x)
        ax.set_xticklabels([f'a{j}' for j in range(len(CANDIDATES))])
        ax.legend()

    plt.tight_layout()
    plt.show()
